In [2]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np
from typing import List, Dict, Tuple

print("Day 15 - Cross-Encoder Re-ranking")

# Bi-encoder -- fast, used for retrieval
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")

# Cross-encoder -- slower, used for re-ranking
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Bi-encoder loaded")
print("Cross-encoder loader")

Day 15 - Cross-Encoder Re-ranking


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Bi-encoder loaded
Cross-encoder loader


In [4]:
print("=== Bi-Encoder vs Cross-Encoder ===\n")

query = "how does retrieval work in RAG?"

chunks = [
    "RAGAs evaluates faithfulness by checking if answers are grounded in context",
    "Hybrid search combines BM25 and vector search merged with RRF for retrieval",
    "Cross-encoder re-rankers process query and document together for precision",
    "FastAPI handles authentication and async request routing for the backend",
    "BM25 ranks documents based on keyword frequency and inverse document frequency"
]

# ----- Bi-encoder approach ----
# Encodes query and documents seperately
# Then computes cosine similarity between vectors
print("=== Bi-Encoder (how your retriever works) ===")
query_embedding = bi_encoder.encode(query)
chunk_embeddings = bi_encoder.encode(chunks)

# Cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
bi_scores = cosine_similarity([query_embedding], chunk_embeddings)[0]

print(f"Query encoded separately: shape {query_embedding.shape}")
print(f"Chunks encoded separately: shape {chunk_embeddings.shape}")
print("\nBi-encoder scores:")
bi_ranked = sorted(enumerate(bi_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(bi_ranked):
    print(f"  Rank {rank+1}: [{score:.4f}] {chunks[idx][:60]}...")

# ---- Cross-encoder approach ---
# Encodes query AND document TOGETHER
# Read full interaction between them
print("\n=== Cross-Encoder (how re-ranking works) ===")
query_chunk_pairs = [[query, chunk] for chunk in chunks]
cross_scores = cross_encoder.predict(query_chunk_pairs)

print(f"Query+chunk pairs evaluated: {len(query_chunk_pairs)}")
print("\nCross-encoder scores:")
cross_ranked = sorted(enumerate(cross_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(cross_ranked):
    print(f"  Rank {rank+1}: [{score:.4f}] {chunks[idx][:60]}...")
    
print("\n=== Key Difference ===")
print("Bi-encoder:   query → vector, doc → vector, compare separately")
print("Cross-encoder: [query + doc] → relevance score, evaluated together")
print("Cross-encoder is slower but far more accurate")




=== Bi-Encoder vs Cross-Encoder ===

=== Bi-Encoder (how your retriever works) ===
Query encoded separately: shape (384,)
Chunks encoded separately: shape (5, 384)

Bi-encoder scores:
  Rank 1: [0.3361] RAGAs evaluates faithfulness by checking if answers are grou...
  Rank 2: [0.1971] Hybrid search combines BM25 and vector search merged with RR...
  Rank 3: [0.1744] Cross-encoder re-rankers process query and document together...
  Rank 4: [0.0990] BM25 ranks documents based on keyword frequency and inverse ...
  Rank 5: [0.0827] FastAPI handles authentication and async request routing for...

=== Cross-Encoder (how re-ranking works) ===
Query+chunk pairs evaluated: 5

Cross-encoder scores:
  Rank 1: [-3.4498] RAGAs evaluates faithfulness by checking if answers are grou...
  Rank 2: [-9.8614] Hybrid search combines BM25 and vector search merged with RR...
  Rank 3: [-11.3703] Cross-encoder re-rankers process query and document together...
  Rank 4: [-11.4657] FastAPI handles authenticat

In [5]:
print("=== Clear Re-ranking Demonstration ===\n")

query = "what is the capital of france?"

# Mix of relevant and irrelevant chunks 
test_chunks = [
    "Python is a programming language used for data science and ML",
    "Paris is the capital city of France and its largest city",
    "Machine learning model require large amounts of training data",
    "France is a country in western Europe. Its capital is Paris",
    "Neural networks are inspired by the human brain strucure"
]

# Bi-encoder
query_emb = bi_encoder.encode(query)
chunk_embs = bi_encoder.encode(test_chunks)
bi_scores = cosine_similarity([query_emb], chunk_embs)[0]

print("Bi-encoder ranking:")
bi_ranked = sorted(enumerate(bi_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(bi_ranked):
    print(f"  Rank {rank+1}: [{score:.4f}] {test_chunks[idx][:60]}...")

# Cross-encoder
pairs = [[query, chunk] for chunk in test_chunks]
cross_encoder = cross_encoder.predict(pairs)

print("\nCross-encoder ranking:")
cross_ranked = sorted(enumerate(cross_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(cross_ranked):
    print(f"  Rank {rank+1}: [{score:.4f}] {test_chunks[idx][:60]}...")

print("\nExpected: Paris/France chunks should rank 1 and 2")

=== Clear Re-ranking Demonstration ===

Bi-encoder ranking:
  Rank 1: [0.8017] France is a country in western Europe. Its capital is Paris...
  Rank 2: [0.7517] Paris is the capital city of France and its largest city...
  Rank 3: [0.1028] Python is a programming language used for data science and M...
  Rank 4: [0.0458] Machine learning model require large amounts of training dat...
  Rank 5: [0.0081] Neural networks are inspired by the human brain strucure...

Cross-encoder ranking:
  Rank 1: [-3.4498] Python is a programming language used for data science and M...
  Rank 2: [-9.8614] Paris is the capital city of France and its largest city...
  Rank 3: [-11.3703] Machine learning model require large amounts of training dat...
  Rank 4: [-11.4657] France is a country in western Europe. Its capital is Paris...
  Rank 5: [-11.4766] Neural networks are inspired by the human brain strucure...

Expected: Paris/France chunks should rank 1 and 2


In [7]:
from sentence_transformers import CrossEncoder

# Reload cross encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Test immediately
test_pairs = [
    ["what is the capital of France?", "Paris is the capital city of France"],
    ["what is the capital of France?", "Python is a programming language"]
]

scores = cross_encoder.predict(test_pairs)
print(f"Paris chunk score: {scores[0]:.4f}")
print(f"Python chunk score: {scores[1]:.4f}")
print(f"Paris should score much higher than Python")

Paris chunk score: 7.7254
Python chunk score: -11.0701
Paris should score much higher than Python


In [8]:
print("=== Correct Re-ranking Demonstration ===\n")

query = "how does retrieval work in RAG? "

chunks = [
     "RAGAs evaluates faithfulness by checking if answers are grounded in context",
    "Hybrid search combines BM25 and vector search merged with RRF for retrieval",
    "Cross-encoder re-rankers process query and document together for precision",
    "FastAPI handles authentication and async request routing for the backend",
    "BM25 ranks documents based on keyword frequency and inverse document frequency"
]

# Bi-encoder
query_emb = bi_encoder.encode(query)
chunk_embs = bi_encoder.encode(chunks)
bi_scores = cosine_similarity([query_emb], chunk_embs)[0]

print("Bi-encoder ranking:")
bi_ranked = sorted(enumerate(bi_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(bi_ranked):
    print(f"  Rank {rank+1}: [{score:.4f}] {chunks[idx][:60]}...")

# Cross-encoder - use different variable name for scores
ce_pairs = [[query, chunk] for chunk in chunks]
ce_scores = cross_encoder.predict(ce_pairs)

print("\nCross-encoder ranking:")
ce_ranked = sorted(enumerate(ce_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(ce_ranked):
    print(f"  Rank {rank+1}: [{score:.4f}] {chunks[idx][:60]}...")

#  Show ranking changes
print("\n=== Ranking Changes ===")
bi_order = [idx for idx, _ in bi_ranked]
ce_order = [idx for idx, _ in ce_ranked]

for ce_rank, idx in enumerate(ce_order):
    bi_rank = bi_order.index(idx)
    change = bi_rank - ce_rank
    arrow = "^"* change if change > 0 else "↓" * abs(change) if change < 0 else "->"
    print(f"   {arrow} '{chunks[idx][:45]}...' (bi:{bi_rank+1} -> ce:{ce_rank+1})")

=== Correct Re-ranking Demonstration ===

Bi-encoder ranking:
  Rank 1: [0.3361] RAGAs evaluates faithfulness by checking if answers are grou...
  Rank 2: [0.1971] Hybrid search combines BM25 and vector search merged with RR...
  Rank 3: [0.1744] Cross-encoder re-rankers process query and document together...
  Rank 4: [0.0990] BM25 ranks documents based on keyword frequency and inverse ...
  Rank 5: [0.0827] FastAPI handles authentication and async request routing for...

Cross-encoder ranking:
  Rank 1: [-3.4498] RAGAs evaluates faithfulness by checking if answers are grou...
  Rank 2: [-9.8614] Hybrid search combines BM25 and vector search merged with RR...
  Rank 3: [-11.3703] Cross-encoder re-rankers process query and document together...
  Rank 4: [-11.4657] FastAPI handles authentication and async request routing for...
  Rank 5: [-11.4766] BM25 ranks documents based on keyword frequency and inverse ...

=== Ranking Changes ===
   -> 'RAGAs evaluates faithfulness by checking if 

In [9]:
print("=== Re-ranking With Clear Relevance Differences ===\n")

query = "how does BM25 keyword search work?"

# Mix of highly relevant, partially relevant, and irrelevant
chunks = [
    "BM25 ranks documents using term frequency and inverse document frequency. It gives higher scores to documents where query terms appear frequently but are rare across the corpus.",
    "Search engines use various algorithms to retrieve documents from large collections of text data stored in databases.",
    "BM25 stands for Best Match 25. The algorithm penalizes very long documents to normalize scores across different document lengths.",
    "Neural networks process information through layers of interconnected nodes called neurons that transform input signals.",
    "Keyword matching algorithms like BM25 score zero for documents that do not contain any query terms making it precise for exact lookups.",
    "FastAPI is a modern Python web framework for building REST APIs with automatic documentation generation."
]

# Bi-encoder scores
query_emb = bi_encoder.encode(query)
chunk_embs = bi_encoder.encode(chunks)
bi_scores = cosine_similarity([query_emb], chunk_embs)[0]

print("Bi-encoder ranking:")
bi_ranked = sorted(enumerate(bi_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(bi_ranked):
    print(f"  Rank {rank+1}: [{score:.4f}] {chunks[idx][:65]}...")

# Cross-encoder scores
ce_pairs = [[query, chunk] for chunk in chunks]
ce_scores = cross_encoder.predict(ce_pairs)

print("\nCross-encoder ranking:")
ce_ranked = sorted(enumerate(ce_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(ce_ranked):
    print(f"  Rank {rank+1}: [{score:.4f}] {chunks[idx][:65]}...")

# Ranking changes
print("\n=== Ranking Changes ===")
bi_order = [idx for idx, _ in bi_ranked]
ce_order = [idx for idx, _ in ce_ranked]
for ce_rank, idx in enumerate(ce_order):
    bi_rank = bi_order.index(idx)
    change = bi_rank - ce_rank
    arrow = "↑"*change if change > 0 else "↓"*abs(change) if change < 0 else "→"
    print(f"  {arrow} Chunk {idx}: '{chunks[idx][:50]}...'")

=== Re-ranking With Clear Relevance Differences ===

Bi-encoder ranking:
  Rank 1: [0.6360] BM25 ranks documents using term frequency and inverse document fr...
  Rank 2: [0.6359] Keyword matching algorithms like BM25 score zero for documents th...
  Rank 3: [0.6202] BM25 stands for Best Match 25. The algorithm penalizes very long ...
  Rank 4: [0.4802] Search engines use various algorithms to retrieve documents from ...
  Rank 5: [0.1576] FastAPI is a modern Python web framework for building REST APIs w...
  Rank 6: [0.0793] Neural networks process information through layers of interconnec...

Cross-encoder ranking:
  Rank 1: [4.3120] Keyword matching algorithms like BM25 score zero for documents th...
  Rank 2: [3.8277] BM25 ranks documents using term frequency and inverse document fr...
  Rank 3: [2.2961] BM25 stands for Best Match 25. The algorithm penalizes very long ...
  Rank 4: [-9.3817] Search engines use various algorithms to retrieve documents from ...
  Rank 5: [-11.3958] N